# Week 2️⃣ Exercise

Applying the concepts learnt this week incorporating streaming, model switching and Gradio for to build a UI.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Download Ollama 3.2
!ollama pull llama3.2

In [ ]:
# Constants
MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

MODEL_OPTIONS = [
    ("gpt-4o-mini (OpenAI)", MODEL_GPT, "openai"),
    ("llama3.2 (Ollama)", MODEL_LLAMA, "ollama"),
]

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
if api_key and api_key.startswith("sk-") and len(api_key) > 10:
    print("OpenAI API key valid ")
else:
    print("Invalid or missing OpenAI API key.")

openai_client = OpenAI(api_key=api_key)
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

SYSTEM_PROMPT = """You are a AI Agronomist.
Given a plant health related questions, come up with a clear, step-by-step tips to help beginner farmers & plant enthusiasts.
Paraphrase: Break down concepts, highlight key ideas, use relevant link examples, and point out common mistakes.
Keep a friendly, expert tone."""

In [ ]:

def stream_reply(history, user_message, model_label):
    # Resolve model and client from dropdown choice
    label_to_config = {label: (model_id, backend) for label, model_id, backend in MODEL_OPTIONS}
    model_id, backend = label_to_config.get(model_label, MODEL_OPTIONS[0][1:])
    client = openai_client if backend == "openai" else ollama

    # Build message list: system + history + new user message
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user, assistant in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": assistant or ""})
    messages.append({"role": "user", "content": user_message})

    # Stream and yield non-empty content only
    stream = client.chat.completions.create(model=model_id, messages=messages, stream=True)
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            yield content

In [ ]:
def chat(message, history, model_choice):
    if not message or not message.strip():
        return
    full = ""
    for chunk in stream_reply(history, message, model_choice):
        full += chunk
        yield full

In [ ]:
model_dropdown = gr.Dropdown(
    choices = [label for label, _, _ in MODEL_OPTIONS],
    value = MODEL_OPTIONS[0][0],
    label = "Model",
)

demo = gr.ChatInterface(
    chat,
    additional_inputs = [model_dropdown],
    title = "Aggy, AI Agronomist",
    description = "Having plant trouble?. Ask Aggy the AI agronomist for help on any plant related queries you may have. You can switch models with the dropdown option.",
)
demo.launch(share = True)